# LLM Judge

## Setup

In [4]:
import litellm
import os
import mlflow
from backend.constants import LLM_JUDGE, MONITORING_PATH

os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

litellm.api_key = os.environ["OPENROUTER_API_KEY"]
litellm.api_base = "https://openrouter.ai/api/v1"

db_path = MONITORING_PATH / "mlflow.db"
if db_path.exists():
    os.chmod(db_path, 0o666)
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

## Impoort for scorers

In [5]:
from mlflow.genai import evaluate
from mlflow.genai.scorers import scorer
import json

## Dataset and predict_fn

In [6]:
import lancedb
import asyncio
import nest_asyncio
from backend.constants import VECTOR_DB_PATH
from backend.agents import bot_answer

nest_asyncio.apply()

vector_db = lancedb.connect(uri=VECTOR_DB_PATH)
docs = vector_db["LectureTranscript"].to_pandas()

evaluation_dataset = [
    {
        "inputs": {"prompt": row["document_name"]},
        "retrieved_context": [{"content": row["content"]}],
    }
    for _, row in docs.head(2).iterrows()
]

def predict_fn(prompt):
    result = asyncio.get_event_loop().run_until_complete(bot_answer(prompt))
    return result.answer

## Scorers and evaluation

In [7]:
RELEVANCE_PROMPT = """Rate from 1-5 how well the answer addresses the question.
Question: {inputs}
Answer: {outputs}
Respond ONLY with JSON: {{"score": <1-5>}}"""

GROUNDEDNESS_PROMPT = """Rate from 1-5 how factually grounded and helpful the answer is.
Question: {inputs}
Answer: {outputs}
Respond ONLY with JSON: {{"score": <1-5>}}"""


def judge(prompt: str) -> int:
    response = litellm.completion(
        model="openrouter/openai/gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.choices[0].message.content
    try:
        return int(json.loads(text)["score"])
    except Exception:
        return 0


@scorer
def relevance(inputs, outputs):
    return judge(RELEVANCE_PROMPT.format(inputs=inputs, outputs=outputs))


@scorer
def groundedness(inputs, outputs):
    return judge(GROUNDEDNESS_PROMPT.format(inputs=inputs, outputs=outputs))


scorers = [relevance, groundedness]

with mlflow.start_run(run_name="rag_evaluation"):
    results = evaluate(data=evaluation_dataset, predict_fn=predict_fn, scorers=scorers)

results

2026/05/08 12:25:24 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/08 12:25:24 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:03, Remaining: 00:00] [predict_fn: 79%, scorers: 21%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: rag_evaluation
  Run ID: 6175791fdf494e408433efd8db572767

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: 6175791fdf494e408433efd8db572767
  metrics:
    groundedness/mean: 4.0
    relevance/mean: 4.0
  result_df: 2 rows x 14 cols
)